In [1]:
import random

import asyncio
import nest_asyncio

import networkx as nx

from src import (
    Loader,
    estimate_cascade_by_community,
)
from src.heuristics.utils import utility_gap

In [2]:
random.seed(42)
nest_asyncio.apply()

In [3]:
from pathlib import Path

paths_to_networks = Path('data/synthetic/networks')

## Simple Barbasi-Albert Graph

In [4]:
k = 20  # number of seeds to select
alpha = 0  # inequality-aversion parameter
p = 0.1  # edge activation probability
num_sims = 100  # number of simulations

In [5]:
from src import kempe_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')

    seeds = kempe_greedy(
        graph=loaded_graph,
        k=k,
        probability=p,
        num_simulations=num_sims,
    )
    print(f'Final seeds: {seeds}')

    final_frac = estimate_cascade_by_community(
        graph=loaded_graph,
        seeds=seeds,
        probability=0.1,
        num_simulations=500,
    )
    print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in final_frac.items()})}')
    print(f'Utility Gap: {round(utility_gap(final_frac), 2)}')


asyncio.run(main())

Selecting seeds: 100%|██████████| 20/20 [01:47<00:00,  5.37s/it, seeds=20]

Final seeds: {0, 3, 5, 6, 9, 21, 22, 151, 152, 26, 30, 43, 46, 52, 61, 85, 472, 88, 741, 105}
Expected influenced fraction per community: {0: 0.05, 1: 0.09, 2: 0.13, 3: 0.13, 4: 0.07, 5: 0.04, 6: 0.04, 7: 0.09, 8: 0.07, 9: 0.08, 10: 0.03, 11: 0.05, 12: 0.02, 13: 0.03, 14: 0.02, 15: 0.03, 16: 0.03}
Utility Gap: 10.96


In [6]:
from src import welfare_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')

    communities = set(nx.get_node_attributes(loaded_graph, 'community').values())

    seeds = welfare_greedy(
        graph=loaded_graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )

    print(f'Selected seed nodes: {seeds}')

    final_frac = estimate_cascade_by_community(
        graph=loaded_graph,
        seeds=seeds,
        probability=0.1,
        num_simulations=500,
    )
    print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in final_frac.items()})}')
    print(f'Utility Gap: {round(utility_gap(final_frac), 2)}')


asyncio.run(main())

Selecting seeds: 100%|██████████| 20/20 [01:57<00:00,  5.88s/it, seeds=20, influenced={0: 0.04, 1: 0.06, 2: 0.07, 3: 0.07, 4: 0.06, 5: 0.05, 6: 0.05, 7: 0.06, 8: 0.08, 9: 0.07, 10: 0.05, 11: 0.06, 12: 0.06, 13: 0.05, 14: 0.06, 15: 0.05, 16: 0.06}]

Selected seed nodes: {0, 1, 263, 415, 674, 39, 571, 699, 450, 68, 709, 738, 358, 230, 360, 870, 876, 503, 382, 895}
Expected influenced fraction per community: {0: 0.06, 1: 0.08, 2: 0.08, 3: 0.09, 4: 0.05, 5: 0.05, 6: 0.05, 7: 0.06, 8: 0.07, 9: 0.07, 10: 0.04, 11: 0.05, 12: 0.05, 13: 0.05, 14: 0.06, 15: 0.04, 16: 0.05}
Utility Gap: 4.61
